##   embedding模型创建

>  数据来源于Amazon,1000条关于美食的评论,全英文

In [1]:
import pandas as pd
from openai import OpenAI
import tiktoken
import os

## 读取数据集

In [2]:
df = pd.read_csv('./datas/fine_food_reviews_1k.csv')
df.columns

Index(['Unnamed: 0', 'Time', 'ProductId', 'UserId', 'Score', 'Summary',
       'Text'],
      dtype='str')

In [3]:
df.head()

,Unnamed: 0,Time,ProductId,UserId,Score,Summary,Text
0,0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...
1,1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos..."
2,2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...
3,3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...
4,4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...


## 设置表头

去掉未命名的第一0列数据,从第一列直接开始

In [5]:
df   = df[df.columns[1:]]

In [6]:
df.head()

,Time,ProductId,UserId,Score,Summary,Text
0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...
1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos..."
2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...
3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...
4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...


文本数据都要拿来处理,所以我们可以合并两个字段,首先可以进行去空值

In [7]:
df=df.dropna()

In [8]:
df.isna()

,Time,ProductId,UserId,Score,Summary,Text
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False
3,False,False,False,False,False,False
4,False,False,False,False,False,False
...,...,...,...,...,...,...
995,False,False,False,False,False,False
996,False,False,False,False,False,False
997,False,False,False,False,False,False
998,False,False,False,False,False,False


把最后俩字段合并成一个新的字段

In [9]:
df['combined']="Title:"+df.Summary.str.strip()+"; Content"+df.Text.str.strip()

In [10]:
df.head()

,Time,ProductId,UserId,Score,Summary,Text,combined
0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title:where does one start...and stop... with...
1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Title:Arrived in pieces; ContentNot pleased at...
2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...,"Title:It isn't blanc mange, but isn't bad . . ..."
3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...,Title:These also have SALT and it's not sea sa...
4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...,Title:Happy with the product; ContentMy dog wa...


##  生成Embedding之后的向量空间,并且保存

In [12]:
embedding_model="text-embedding-ada-002"

## 指定对文本进行处理的分词器

In [16]:
tokenizer_name= 'cl100k_base'

In [15]:
max_tokens=8191
top_n=1000

In [14]:
df = df.sort_values('Time')
df.drop('Time',axis=1,inplace=True)

##  预处理结束,开始构建向量空间

In [17]:
# 创建一个分词器
tokenizer=tiktoken.get_encoding(encoding_name=tokenizer_name)

控制输入数据的token数量
计算token数量

In [18]:
df['count_token']=df.combined.apply(lambda x: len(tokenizer.encode(x)))

token的数量不能超过官方的阈值,超过了就不要

In [19]:
df = df[df.count_token  <= max_tokens].tail(top_n)
print(len(df))

1000


## 初始化openai客户端

In [21]:
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
client = OpenAI()

In [23]:
def embedding_text(text,model="text-embedding-ada-002"):
    resp = client.embeddings.create(input=text,model=model)
    return resp.data[0].embedding

In [25]:
df['embedding']=df.combined.apply(embedding_text)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

是的，你的理解完全正确！

`df['embedding'] = df.combined.apply(embedding_text)` 这行代码的含义是：

1.  **目标列**：`df.combined` 选中了数据框中的 `combined` 这一列（这是一个 Series，即一列数据）。
2.  **应用函数**：`.apply(embedding_text)` 会遍历 `combined` 列中的**每一个单元格**（每一行）。
3.  **传入参数**：它会将当前遍历到的那个单元格的值（比如某一行具体的文本内容），作为参数传递给 `embedding_text` 函数。
4.  **收集结果**：`embedding_text` 函数处理完后返回的结果（通常是向量列表），会被收集起来。
5.  **赋值**：最后，所有返回的结果组成一个新的列，赋值给 `df['embedding']`。

### 举个简单的例子方便你理解：

假设 `df` 长这样：

| index | combined (文本) |
| :--- | :--- |
| 0 | "今天天气不错" |
| 1 | "熊猫喜欢吃竹子" |
| 2 | "机器学习很有趣" |

假设 `embedding_text` 函数定义如下（模拟）：
```python
def embedding_text(text):
    # 假设这个函数返回文本长度的列表作为模拟向量
    return [len(text), 0.5]
```

执行 `df.combined.apply(embedding_text)` 的过程如下：

1.  **第1轮**：取出 `"今天天气不错"` 传入函数 $\rightarrow$ 返回 `[6, 0.5]`
2.  **第2轮**：取出 `"熊猫喜欢吃竹子"` 传入函数 $\rightarrow$ 返回 `[7, 0.5]`
3.  **第3轮**：取出 `"机器学习很有趣"` 传入函数 $\rightarrow$ 返回 `[7, 0.5]`

最终 `df['embedding']` 就会变成：

| index | combined | embedding (新生成的列) |
| :--- | :--- | :--- |
| 0 | "今天天气不错" | `[6, 0.5]` |
| 1 | "熊猫喜欢吃竹子" | `[7, 0.5]` |
| 2 | "机器学习很有趣" | `[7, 0.5]` |

### 补充提示
在你的 Embedding 场景中，`embedding_text` 函数内部通常会调用类似 `client.embeddings.create(...)` 的代码，把传入的文本转换成向量。`.apply()` 的好处就是不用写 `for` 循环，代码更简洁高效。